# Results-section statistics verification

Recomputes every statistical value claimed in §4 of `main4.tex` and
prints a side-by-side comparison: what the report says, what the
current notebook outputs produce, and a MATCH / MISMATCH flag.

Run top to bottom after any change to the pipeline (classifier,
flower-visit detector, indicator definitions, etc.) so the report
text and the data stay in sync.

**One cell per Results subsection (4.3 -> 4.11).**

Scipy-free: Mann-Whitney U, rank-biserial $r$, Spearman $\rho$,
Benjamini-Hochberg correction, and the bootstrap CI are all
hand-rolled.

In [1]:
# ─── Imports + scipy-free helpers ───
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from math import erf, sqrt
import numpy as np
import pandas as pd

OK   = "MATCH"
BAD  = "MISMATCH"
SOFT = "≈"

def flag(reported, computed, tol=0.01):
    """Decide MATCH / MISMATCH for one numeric claim. tol is absolute tolerance."""
    if reported is None or computed is None: return "?"
    return OK if abs(reported - computed) <= tol else BAD

def mwu(a, b):
    """Mann-Whitney U two-sided, rank-biserial r. Returns (U, p, r) or (nan,nan,nan)."""
    a = np.asarray(a, dtype=float); a = a[~np.isnan(a)]
    b = np.asarray(b, dtype=float); b = b[~np.isnan(b)]
    na, nb = len(a), len(b)
    if na < 2 or nb < 2: return np.nan, np.nan, np.nan
    ranks = pd.Series(np.concatenate([a, b])).rank()
    Ra = ranks[:na].sum()
    U1 = Ra - na*(na+1)/2
    U2 = na*nb - U1
    U  = min(U1, U2)
    mu = na*nb/2
    sd = sqrt(na*nb*(na+nb+1)/12)
    z  = 0.0 if sd == 0 else (U - mu) / sd
    p  = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
    r  = 1 - 2*U / (na*nb)
    return float(U), float(p), float(r)

def spearman(x, y):
    """Spearman rho + two-sided p (normal-approximation)."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]
    n = len(x)
    if n < 3: return np.nan, np.nan, n
    rx = pd.Series(x).rank().values
    ry = pd.Series(y).rank().values
    rho = float(np.corrcoef(rx, ry)[0, 1])
    if abs(rho) >= 0.9999: p = 0.0
    else:
        t = rho * sqrt((n - 2) / (1 - rho * rho))
        p = 2 * (1 - 0.5 * (1 + erf(abs(t) / sqrt(2))))
    return rho, float(p), n

def bh_correct(pvals):
    """Benjamini-Hochberg adjusted p-values."""
    p = np.asarray(pvals, dtype=float)
    n = len(p); order = np.argsort(p)
    adj = np.empty(n)
    cum_min = 1.0
    for rank, idx in enumerate(reversed(order)):
        k = n - rank
        cum_min = min(cum_min, p[idx] * n / k)
        adj[idx] = cum_min
    return adj

def bootstrap_r(on, off, n_boot=2000, seed=42):
    """Pooled rank-biserial r bootstrap CI."""
    rng = np.random.default_rng(seed)
    rs = []
    for _ in range(n_boot):
        on_b  = rng.choice(on,  size=len(on),  replace=True)
        off_b = rng.choice(off, size=len(off), replace=True)
        _, _, r = mwu(on_b, off_b)
        rs.append(r)
    rs = np.array(rs)
    return float(np.percentile(rs, 2.5)), float(np.percentile(rs, 97.5))


## Load all data sources

The pipeline output `data/multi_day_v3/` is the canonical source for
the v3-classifier-derived indicators. Flower-visit detector outputs
live in `data/multi_day/`. Greenhouse temperature is loaded from the
latest `greenhouse_temp_*.csv` file in `data/`.

In [2]:
# ─── Constants ───
CYCLE_ANCHOR   = pd.Timestamp("2026-04-23")
EXCLUDED_DATES = {pd.Timestamp("2026-05-26")}
OFF_OVERRIDE   = {pd.Timestamp("2026-05-29")}

def label_date(d):
    d = pd.Timestamp(d)
    if d in EXCLUDED_DATES: return "EXCLUDED"
    if d in OFF_OVERRIDE:   return "OFF"
    if d < CYCLE_ANCHOR:    return "BASELINE"
    return "ON" if ((d - CYCLE_ANCHOR).days // 3) % 2 == 0 else "OFF"

DATA_ROOT = Path("../../data")
V3_DIR    = Path("data/multi_day_v3")
FV_DIR    = Path("data/multi_day")

ind = pd.read_csv(V3_DIR / "indicators_daily.csv")
ptk = pd.read_csv(V3_DIR / "per_track_indicators.csv", parse_dates=["ts"])
daily = pd.read_csv(V3_DIR / "daily_summary.csv")
fv  = pd.read_csv(FV_DIR / "flower_visits.csv")
fv_s = pd.read_csv(FV_DIR / "flower_visit_summary.csv")

for df in (ind, daily, fv_s):
    df["date_ts"]   = pd.to_datetime(df["date"])
    df["condition"] = df["date_ts"].apply(label_date)
ptk["date_ts"]   = pd.to_datetime(ptk["date"])
ptk["condition"] = ptk["date_ts"].apply(label_date)
ptk["hour"]      = ptk["ts"].dt.hour

# Temperature (greenhouse kastemperatuur sensor)
# File format: 4-sensor interleaved semicolon-CSV, decimal comma.
# Columns 9 & 10 are kastemperatuur (TimeStamp, Value).
gh_files = sorted(DATA_ROOT.glob("greenhouse_temp_*.csv"))
temp_daily = None
if gh_files:
    raw = pd.read_csv(gh_files[-1], sep=";", header=None, skiprows=4,
                      dtype=str)
    ts  = pd.to_datetime(raw[9],  errors="coerce", utc=True).dt.tz_convert(None)
    val = pd.to_numeric(raw[10].str.replace(",", ".", regex=False), errors="coerce")
    gh = pd.DataFrame({"ts": ts, "temp_c": val}).dropna()
    gh["date_str"] = gh["ts"].dt.strftime("%Y-%m-%d")
    gh["hour"]     = gh["ts"].dt.hour
    daytime = gh[gh["hour"].between(8, 18)]
    temp_daily = (daytime.groupby("date_str")
                          .agg(temp_mean=("temp_c","mean"))
                          .reset_index().rename(columns={"date_str":"date"}))

print(f"indicators_daily   : {ind.shape}  ({ind.date.min()} -> {ind.date.max()})")
print(f"per_track_indicators: {ptk.shape}")
print(f"daily_summary      : {daily.shape}")
print(f"flower_visits      : {fv.shape}")
print(f"temperature_daily  : {temp_daily.shape if temp_daily is not None else 'NOT LOADED'}")
print()
print("Coverage:")
print(ind.groupby(["system_id","condition"]).size().unstack(fill_value=0))


indicators_daily   : (82, 23)  (2026-04-13 -> 2026-05-28)
per_track_indicators: (66239, 20)
daily_summary      : (82, 15)
flower_visits      : (3193, 19)
temperature_daily  : (55, 2)

Coverage:
condition  BASELINE  OFF  ON
system_id                   
900              10   17  18
939               2   17  18


## §4.3 — Flower-visit detection outcome

Report (line ≈859):
- "Applied to system 900 across $n = 35$ day-system rows"
- "1,542 confirmed flower visits"
- "663 (43.0 %) of the within-track type"
- "879 (57.0 %) of the cross-track type"
- "median of 2.93 s and an inter-quartile range of 1.44–10.39 s"


In [3]:
# ─── §4.3 verification ───
fv_900 = fv[fv["system_id"] == 900].copy()
if "date" not in fv_900.columns:
    fv_900["date"] = fv_900["wall_start"].astype(str).str[:10]
fv_900["date"] = pd.to_datetime(fv_900["date"]).dt.strftime("%Y-%m-%d")
fv_exp = fv_900[fv_900["date"] >= "2026-04-23"]

n_visits  = len(fv_exp)
n_within  = int((fv_exp["source"]=="within").sum())
n_cross   = int((fv_exp["source"]=="cross").sum())
pct_within = 100 * n_within / n_visits
pct_cross  = 100 * n_cross  / n_visits
n_day_sys  = fv_exp.groupby(["date","system_id"]).size().shape[0]
med_dur    = float(fv_exp["duration_s"].median())
q25, q75   = float(fv_exp["duration_s"].quantile(0.25)), float(fv_exp["duration_s"].quantile(0.75))

print(f"{'quantity':30s}  {'report':>10s}  {'computed':>10s}  status")
print("-"*70)
print(f"{'n day-system rows':30s}  {35:>10}  {n_day_sys:>10}  {flag(35, n_day_sys, 0)}")
print(f"{'total visits':30s}  {1542:>10}  {n_visits:>10}  {flag(1542, n_visits, 0)}")
print(f"{'within-track count':30s}  {663:>10}  {n_within:>10}  {flag(663, n_within, 0)}")
print(f"{'within-track %':30s}  {43.0:>10.1f}  {pct_within:>10.1f}  {flag(43.0, pct_within, 0.1)}")
print(f"{'cross-track count':30s}  {879:>10}  {n_cross:>10}  {flag(879, n_cross, 0)}")
print(f"{'cross-track %':30s}  {57.0:>10.1f}  {pct_cross:>10.1f}  {flag(57.0, pct_cross, 0.1)}")
print(f"{'median duration (s)':30s}  {2.93:>10.2f}  {med_dur:>10.2f}  {flag(2.93, med_dur, 0.02)}")
print(f"{'IQR low (s)':30s}  {1.44:>10.2f}  {q25:>10.2f}  {flag(1.44, q25, 0.02)}")
print(f"{'IQR high (s)':30s}  {10.39:>10.2f}  {q75:>10.2f}  {flag(10.39, q75, 0.02)}")


quantity                            report    computed  status
----------------------------------------------------------------------
n day-system rows                       35          35  MATCH
total visits                          1542        1542  MATCH
within-track count                     663         663  MATCH
within-track %                        43.0        43.0  MATCH
cross-track count                      879         879  MATCH
cross-track %                         57.0        57.0  MATCH
median duration (s)                   2.93        2.93  MATCH
IQR low (s)                           1.44        1.44  MATCH
IQR high (s)                         10.39       10.39  MATCH


## §4.4 — Daily activity curves and temperature

Report (line ≈893):
- Spearman $\rho = -0.59$, $p < 0.001$, $n = 36$ (greenhouse temp vs exit+return count, sys 900, ON+OFF only)
- ON-vs-OFF rank-biserial $r = +0.03$ for `exit_return_count`


In [4]:
# ─── §4.4 verification ───
# Build exit_return_count per (date, system_id) from indicators_daily
sub = ind[ind.condition.isin(["ON","OFF"])].copy()
sub["exit_return_count"] = sub["n_exit_v3"] + sub["n_return_v3"]
sub_900 = sub[sub.system_id == 900].copy()

# Spearman temperature vs exit+return
sub_900["date_str"] = sub_900["date_ts"].dt.strftime("%Y-%m-%d")
joined = sub_900.merge(temp_daily, left_on="date_str", right_on="date", how="inner")
rho, p, n = spearman(joined["temp_mean"].values, joined["exit_return_count"].values)

# MWU r for exit_return_count, ON vs OFF, sys 900
on  = sub_900[sub_900.condition=="ON" ]["exit_return_count"]
off = sub_900[sub_900.condition=="OFF"]["exit_return_count"]
U, p_mwu, r_rb = mwu(on, off)

print("§4.4 statistical values:\n")
print(f"  Spearman rho (temp vs exit+return, sys 900):")
print(f"    report   : rho = -0.59, p < 0.001, n = 36")
print(f"    computed : rho = {rho:+.3f}, p = {p:.4f}, n = {n}")
print(f"    status   : rho {flag(-0.59, rho, 0.02)},  n {flag(36, n, 0)}")
print()
print(f"  MWU rank-biserial r, exit_return_count, sys 900:")
print(f"    report   : r = +0.03")
print(f"    computed : r = {r_rb:+.3f}, p = {p_mwu:.3f}, "
      f"n_ON = {len(on)}, n_OFF = {len(off)}")
print(f"    status   : r {flag(+0.03, r_rb, 0.02)}")


§4.4 statistical values:

  Spearman rho (temp vs exit+return, sys 900):
    report   : rho = -0.59, p < 0.001, n = 36
    computed : rho = -0.521, p = 0.0005, n = 35
    status   : rho MISMATCH,  n MISMATCH

  MWU rank-biserial r, exit_return_count, sys 900:
    report   : r = +0.03
    computed : r = +0.056, p = 0.779, n_ON = 18, n_OFF = 17
    status   : r MISMATCH


## §4.5 — Behavioural anchors

Report (lines 909–911):
- sys 900 peak_hour: $r = +0.23$, $p = 1.00$  (ON: 10:00, OFF: 11:00)
- sys 900 active_hours: $r = +0.06$, $p = 0.312$
- sys 939 (both anchors): $|r| \le 0.13$, both $p \ge 0.50$


In [12]:
# ─── §4.5 verification ───
# Match the anchor definitions from exposure_analysis_v3.ipynb cell 17:
#   active_hours = number of hours where exits/hour >= ACTIVITY_THRESHOLD
#   peak_hour    = hour with the max number of exits
ACTIVITY_THRESHOLD = 5
exits = ptk[ptk["hive_exit_v3"] == True].copy() if "hive_exit_v3" in ptk.columns else ptk[ptk.get("is_exit", False) == True].copy()

anchors_rows = []
for (d, sid), grp in exits.groupby(["date", "system_id"]):
    counts = grp.groupby("hour").size()           # exits per hour
    if counts.empty:
        continue
    anchors_rows.append({
        "date":         pd.Timestamp(d).strftime("%Y-%m-%d"),
        "system_id":    sid,
        "peak_hour":    int(counts.idxmax()),
        "active_hours": int((counts >= ACTIVITY_THRESHOLD).sum()),
    })
anchors = pd.DataFrame(anchors_rows)
ind["date_str"] = ind["date_ts"].dt.strftime("%Y-%m-%d")
ind_anc = ind.merge(anchors, left_on=["date_str","system_id"], right_on=["date","system_id"], how="left")
sub_anc = ind_anc[ind_anc.condition.isin(["ON","OFF"])]

print("§4.5 statistical values:\n")
print(f"  {'system':>4}  {'anchor':14s}  {'med ON':>7}  {'med OFF':>8}  {'r':>7}  {'p':>7}")
print("  " + "-"*60)
expectations = {(900,"peak_hour"):(+0.23, 1.00), (900,"active_hours"):(+0.06, 0.312),
                (939,"peak_hour"):(None,None), (939,"active_hours"):(None,None)}
for sid in [900, 939]:
    s = sub_anc[sub_anc.system_id == sid]
    for col in ["peak_hour", "active_hours"]:
        on  = s[s.condition=="ON" ][col].dropna()
        off = s[s.condition=="OFF"][col].dropna()
        if len(on)<2 or len(off)<2: continue
        U, p, r = mwu(on, off)
        rep_r, rep_p = expectations.get((sid, col), (None, None))
        status_r = flag(rep_r, r, 0.05) if rep_r is not None else "(not pinned)"
        status_p = flag(rep_p, p, 0.10) if rep_p is not None else ""
        print(f"  {sid:>4}  {col:14s}  {on.median():>7.1f}  {off.median():>8.1f}  "
              f"{r:>+7.3f}  {p:>7.3f}   {status_r} {status_p}")

# Also flag the prose: "ON: 10:00, OFF: 11:00" for sys 900 peak hour
s900 = sub_anc[sub_anc.system_id == 900]
on_med  = s900[s900.condition=="ON" ]["peak_hour"].median()
off_med = s900[s900.condition=="OFF"]["peak_hour"].median()
print(f"\n  Prose check: sys 900 peak_hour medians:")
print(f"    report   : ON = 10:00, OFF = 11:00 (1h shift)")
print(f"    computed : ON = {on_med:.1f}:00, OFF = {off_med:.1f}:00 "
      f"(shift = {off_med - on_med:+.0f}h)")
print(f"    status   : {'MATCH' if abs((off_med - on_med) - 1) < 0.5 else 'MISMATCH'}")


§4.5 statistical values:

  system  anchor           med ON   med OFF        r        p
  ------------------------------------------------------------
   900  peak_hour          10.0      10.0   +0.003    0.987   MISMATCH MATCH
   900  active_hours        4.0       5.0   +0.193    0.330   MISMATCH MATCH
   939  peak_hour          11.0      11.0   +0.078    0.692   (not pinned) 
   939  active_hours        3.0       3.0   +0.114    0.564   (not pinned) 

  Prose check: sys 900 peak_hour medians:
    report   : ON = 10:00, OFF = 11:00 (1h shift)
    computed : ON = 10.0:00, OFF = 10.0:00 (shift = +0h)
    status   : MISMATCH


## §4.6 — Behavioural indicator dashboard

12 Mann–Whitney tests (6 indicators × 2 systems), reported as
rank-biserial $r$ and raw $p$, with BH-adjusted $p$ across the
12-test family. The report claims:
- sys 939 ifi_cv: $r = -0.31$, raw $p = 0.117$
- sys 900 ifi_cv: $r = -0.14$, $p = 0.48$
- No cell survives BH correction (all $p_{\text{BH}} \ge 0.05$)
- Max $|r|$ on sys 900: "negligible ($|r| \le 0.14$)"
- Max $|r|$ on sys 939: "$|r| \le 0.13$ (anchors), and dashboard $|r|$ small except ifi_cv = $-0.31$"


In [6]:
# ─── §4.6 verification ───
INDICATORS = ["neg_exit_count", "neg_re_ratio", "path_tortuosity", "ifi_cv",
              "mean_handling_time_s", "n_distinct_flowers"]
sub = ind[ind.condition.isin(["ON","OFF"])].copy()

rows = []
for sid in [900, 939]:
    for col in INDICATORS:
        on  = sub[(sub.system_id==sid) & (sub.condition=="ON" )][col].dropna()
        off = sub[(sub.system_id==sid) & (sub.condition=="OFF")][col].dropna()
        U, p, r = mwu(on, off)
        rows.append({"system":sid, "indicator":col, "n_ON":len(on), "n_OFF":len(off),
                     "med_ON":on.median(), "med_OFF":off.median(), "r":r, "p_raw":p})

table = pd.DataFrame(rows)
table["p_BH"] = bh_correct(table["p_raw"].values)

print("§4.6 — full 12-cell dashboard:\n")
with pd.option_context("display.max_rows", None, "display.width", 160,
                       "display.float_format", "{:+.3f}".format):
    show = table.copy()
    show["star"] = np.where(show["r"].abs() > 0.3, "*", "")
    print(show[["system","indicator","n_ON","n_OFF","med_ON","med_OFF","r","p_raw","p_BH","star"]].to_string(index=False))

# specific report claims
def lookup(sid, col):
    row = table[(table.system==sid)&(table.indicator==col)].iloc[0]
    return float(row["r"]), float(row["p_raw"])

r_939_ifi, p_939_ifi = lookup(939, "ifi_cv")
r_900_ifi, p_900_ifi = lookup(900, "ifi_cv")
max_r_900 = float(table[table.system==900]["r"].abs().max())
max_r_939 = float(table[table.system==939]["r"].abs().max())
any_bh = bool((table["p_BH"] < 0.05).any())

print("\nReport-vs-computed checks:")
print(f"  sys 939 ifi_cv r        : report -0.31, computed {r_939_ifi:+.3f}  {flag(-0.31, r_939_ifi, 0.05)}")
print(f"  sys 939 ifi_cv p (raw)  : report  0.117, computed {p_939_ifi:.3f}  {flag(0.117, p_939_ifi, 0.05)}")
print(f"  sys 900 ifi_cv r        : report -0.14, computed {r_900_ifi:+.3f}  {flag(-0.14, r_900_ifi, 0.05)}")
print(f"  sys 900 ifi_cv p        : report  0.48, computed {p_900_ifi:.3f}   {flag(0.48, p_900_ifi, 0.05)}")
print(f"  sys 900 max |r|         : report  ≤ 0.14, computed {max_r_900:.3f}  "
      f"{'MATCH' if max_r_900 <= 0.14+0.02 else 'MISMATCH'}")
print(f"  sys 939 max |r|         : computed {max_r_939:.3f}")
print(f"  any p_BH < 0.05         : report  no, computed {'yes' if any_bh else 'no'}  "
      f"{'MATCH' if not any_bh else 'MISMATCH'}")


§4.6 — full 12-cell dashboard:

 system            indicator  n_ON  n_OFF  med_ON  med_OFF      r  p_raw   p_BH star
    900       neg_exit_count    18     17 -59.500  -61.000 +0.007 +0.974 +0.974     
    900         neg_re_ratio    18     17  -1.624   -1.580 +0.137 +0.488 +0.974     
    900      path_tortuosity    18     17  +2.291   +2.293 +0.026 +0.895 +0.974     
    900               ifi_cv    18     17  +2.175   +2.010 +0.098 +0.621 +0.974     
    900 mean_handling_time_s    18     17  +6.620   +6.648 +0.013 +0.947 +0.974     
    900   n_distinct_flowers    18     17  +5.500   +5.000 +0.056 +0.779 +0.974     
    939       neg_exit_count    18     17 -27.500  -29.000 +0.111 +0.575 +0.974     
    939         neg_re_ratio    18     17  -1.566   -1.739 +0.059 +0.766 +0.974     
    939      path_tortuosity    18     17  +2.302   +2.293 +0.026 +0.895 +0.974     
    939               ifi_cv    17     17  +1.401   +1.290 +0.190 +0.344 +0.974     
    939 mean_handling_time_s    1

## §4.7 — Date-window sensitivity (system 900)

Report claims (around line ≈940):
- `path_tortuosity` shows the largest window-sensitivity: $\Delta r = +0.21$ (effect drops when last hot ON block trimmed)
- Direct temperature test: Spearman $\rho = +0.78$, $p < 0.001$ for `path_tortuosity` vs greenhouse temperature on sys 900


In [7]:
# ─── §4.7 verification ───
# Spearman: path_tortuosity vs greenhouse temperature, sys 900, ON/OFF days only
sub_900 = ind[(ind.system_id==900) & (ind.condition.isin(["ON","OFF"]))].copy()
sub_900["date_str"] = sub_900["date_ts"].dt.strftime("%Y-%m-%d")
joined = sub_900.merge(temp_daily, left_on="date_str", right_on="date", how="inner")
rho_tort, p_tort, n_tort = spearman(joined["temp_mean"].values, joined["path_tortuosity"].values)

# Window sensitivity: r on full window vs r on date <= 5/16
full = sub_900
trim = sub_900[sub_900["date_ts"] <= pd.Timestamp("2026-05-16")]
def r_for(df, col):
    on  = df[df.condition=="ON" ][col].dropna()
    off = df[df.condition=="OFF"][col].dropna()
    _,_,r = mwu(on, off)
    return r, len(on), len(off)

r_full,  n_on_full,  n_off_full  = r_for(full, "path_tortuosity")
r_trim,  n_on_trim,  n_off_trim  = r_for(trim, "path_tortuosity")
delta_r  = r_full - r_trim

print("§4.7 statistical values:\n")
print(f"  Spearman rho (temp vs path_tortuosity, sys 900):")
print(f"    report   : rho = +0.78, p < 0.001")
print(f"    computed : rho = {rho_tort:+.3f}, p = {p_tort:.4f}, n = {n_tort}")
print(f"    status   : {flag(+0.78, rho_tort, 0.03)}")
print()
print(f"  Window sensitivity, path_tortuosity ON-vs-OFF r:")
print(f"    full window         : r = {r_full:+.3f}  (n_ON={n_on_full}, n_OFF={n_off_full})")
print(f"    trimmed to <=5/16   : r = {r_trim:+.3f}  (n_ON={n_on_trim}, n_OFF={n_off_trim})")
print(f"    delta r            : {delta_r:+.3f}  (report says +0.21)  "
      f"{flag(+0.21, delta_r, 0.03)}")


§4.7 statistical values:

  Spearman rho (temp vs path_tortuosity, sys 900):
    report   : rho = +0.78, p < 0.001
    computed : rho = +0.693, p = 0.0000, n = 35
    status   : MISMATCH

  Window sensitivity, path_tortuosity ON-vs-OFF r:
    full window         : r = +0.026  (n_ON=18, n_OFF=17)
    trimmed to <=5/16   : r = +0.194  (n_ON=12, n_OFF=12)
    delta r            : -0.168  (report says +0.21)  MISMATCH


## §4.8 — Heading dispersion (Δ R permutation test)

Report (around line ≈974):
- sys 939 entry: $\Delta R = +0.023$, $p = 0.020$
- sys 939 exit:  $\Delta R = +0.028$, $p = 0.007$
- sys 900 entry/exit: both non-significant ($p \ge 0.05$)


In [8]:
# ─── §4.8 verification (sys 939 + sys 900 heading delta-R permutation) ───
GEOM_CSV = FV_DIR / "track_geometry.csv"
if not GEOM_CSV.exists():
    print(f"!! {GEOM_CSV} not found — skipping §4.8 check.")
else:
    geom = pd.read_csv(GEOM_CSV)
    geom["date_ts"]   = pd.to_datetime(geom["date"])
    geom["condition"] = geom["date_ts"].apply(label_date)
    geom = geom[geom.condition.isin(["ON","OFF"])]

    def R_of(angles_deg):
        rad = np.radians(angles_deg.dropna().values)
        if len(rad) < 3: return np.nan
        return float(np.abs(np.mean(np.exp(1j*rad))))

    def perm_test(angles, labels, n_perm=5000, seed=42):
        rng = np.random.default_rng(seed)
        on  = angles[labels == "ON"]
        off = angles[labels == "OFF"]
        if len(on)<3 or len(off)<3: return np.nan, np.nan
        R_on, R_off = R_of(on), R_of(off)
        delta = R_on - R_off
        pooled = np.concatenate([on.dropna(), off.dropna()])
        n_on = on.dropna().size
        ge = 0
        for _ in range(n_perm):
            rng.shuffle(pooled)
            a, b = pooled[:n_on], pooled[n_on:]
            ra = float(np.abs(np.mean(np.exp(1j*np.radians(a)))))
            rb = float(np.abs(np.mean(np.exp(1j*np.radians(b)))))
            if abs(ra - rb) >= abs(delta): ge += 1
        return float(delta), (ge + 1) / (n_perm + 1)

    print("§4.8 statistical values (permutation test, 5000 shuffles):\n")
    expectations = {(939,"entry"):(+0.023, 0.020), (939,"exit"):(+0.028, 0.007),
                    (900,"entry"):(None, None),    (900,"exit"):(None, None)}
    for sid in [900, 939]:
        for col, name in [("heading_entry_deg","entry"), ("heading_exit_deg","exit")]:
            s = geom[geom.system_id == sid]
            d, p = perm_test(s[col], s["condition"], n_perm=5000)
            rep_d, rep_p = expectations[(sid, name)]
            status = "(not pinned)" if rep_d is None else flag(rep_d, d, 0.01) + " / " + flag(rep_p, p, 0.02)
            print(f"  sys {sid}  {name:5s}  delta R = {d:+.3f},  p = {p:.4f}   {status}")


§4.8 statistical values (permutation test, 5000 shuffles):

  sys 900  entry  delta R = +0.011,  p = 0.1206   (not pinned)
  sys 900  exit   delta R = +0.008,  p = 0.2783   (not pinned)
  sys 939  entry  delta R = +0.005,  p = 0.5973   MISMATCH / MISMATCH
  sys 939  exit   delta R = +0.020,  p = 0.0438   MATCH / MISMATCH


## §4.9 — Accumulation and drift

Report (around line ≈994):
- Day-1-ON vs Day-3-ON within each ON block: no indicator survives ($n = 6$ per group, all $|r| \le 0.31$)
- Day-3-OFF vs Day-1-ON transition: no indicator survives ($n = 6$ per group, no $p < 0.05$)


In [9]:
# ─── §4.9 verification ───
def day_in_on_block(d):
    d = pd.Timestamp(d)
    if d < CYCLE_ANCHOR: return np.nan
    days_in = (d - CYCLE_ANCHOR).days
    if (days_in // 3) % 2 != 0: return np.nan  # OFF
    return (days_in % 3) + 1                    # 1, 2, 3

def day_in_off_block(d):
    d = pd.Timestamp(d)
    if d < CYCLE_ANCHOR: return np.nan
    days_in = (d - CYCLE_ANCHOR).days
    if (days_in // 3) % 2 == 0: return np.nan  # ON
    return (days_in % 3) + 1

ind["day_in_on"]  = ind["date_ts"].apply(day_in_on_block)
ind["day_in_off"] = ind["date_ts"].apply(day_in_off_block)

print("§4.9 — Day-1-ON vs Day-3-ON (accumulation):\n")
INDICATORS = ["neg_exit_count","neg_re_ratio","path_tortuosity","ifi_cv",
              "mean_handling_time_s","n_distinct_flowers"]
max_abs_r = 0
for sid in [900, 939]:
    s = ind[(ind.system_id==sid) & (ind.condition=="ON")]
    for col in INDICATORS:
        d1 = s[s.day_in_on == 1][col].dropna()
        d3 = s[s.day_in_on == 3][col].dropna()
        if len(d1)<2 or len(d3)<2: continue
        U, p, r = mwu(d1, d3)
        max_abs_r = max(max_abs_r, abs(r))
        if abs(r) > 0.3 or p < 0.05:
            print(f"  sys {sid}  {col:25s}  r = {r:+.3f}  p = {p:.3f}  (flagged)")
print(f"  max |r| across all 12 day-1 vs day-3 comparisons = {max_abs_r:.3f}  "
      f"(report says <= 0.31)  {'MATCH' if max_abs_r <= 0.31+0.02 else 'MISMATCH'}")

print("\n§4.9 — Day-3-OFF vs Day-1-ON (drift / transition):\n")
max_abs_r = 0
any_sig = False
for sid in [900, 939]:
    s_off = ind[(ind.system_id==sid) & (ind.condition=="OFF") & (ind.day_in_off==3)]
    s_on  = ind[(ind.system_id==sid) & (ind.condition=="ON" ) & (ind.day_in_on==1)]
    for col in INDICATORS:
        a = s_off[col].dropna(); b = s_on[col].dropna()
        if len(a)<2 or len(b)<2: continue
        U, p, r = mwu(a, b)
        max_abs_r = max(max_abs_r, abs(r))
        if p < 0.05: any_sig = True
        if abs(r) > 0.3 or p < 0.05:
            print(f"  sys {sid}  {col:25s}  r = {r:+.3f}  p = {p:.3f}  (flagged)")
print(f"  max |r| = {max_abs_r:.3f}, any p<0.05 = {any_sig}  "
      f"(report says no significant transitions)  {'MATCH' if not any_sig else 'MISMATCH'}")


§4.9 — Day-1-ON vs Day-3-ON (accumulation):

  sys 939  neg_re_ratio               r = +0.389  p = 0.262  (flagged)
  max |r| across all 12 day-1 vs day-3 comparisons = 0.389  (report says <= 0.31)  MISMATCH

§4.9 — Day-3-OFF vs Day-1-ON (drift / transition):

  sys 900  mean_handling_time_s       r = +0.389  p = 0.262  (flagged)
  sys 939  neg_exit_count             r = +0.389  p = 0.262  (flagged)
  sys 939  neg_re_ratio               r = +0.444  p = 0.200  (flagged)
  max |r| = 0.444, any p<0.05 = False  (report says no significant transitions)  MATCH


## §4.10 — Dose-response analysis

Report (around line ≈1063):
- path_tortuosity vs received power: Spearman $\rho = -0.71$, $p < 0.001$
- v3 exits vs received power: $\rho = +0.56$, $p = 0.007$
- mean_handling_time vs received power: $\rho = +0.59$, $p = 0.003$
- Daily greenhouse temp vs activity: $\rho = -0.59$ (from §4.4)


In [10]:
# ─── §4.10 verification ───
# Dose-response uses received power (mean_dbm) vs indicator, on ON days only.
sub_on = ind[(ind.condition=="ON") & (ind.system_id==900)].copy()
sub_on["exit_return_count"] = sub_on["n_exit_v3"] + sub_on["n_return_v3"]

n_dbm = int(sub_on["mean_dbm"].notna().sum())
print("§4.10 — Spearman rho (received power vs indicators, sys 900 ON days):\n")
if n_dbm == 0:
    print(f"  !! WARNING: mean_dbm column is entirely NaN in indicators_daily.")
    print(f"     The dBm loader cells in multi_day_pipeline_v3.ipynb need to be")
    print(f"     re-run (they probably failed or were skipped when the v3 pipeline")
    print(f"     was last re-run). Until that happens the §4.10 numbers cannot be")
    print(f"     verified from this notebook.")
    print(f"  report values (cannot be checked yet):")
    for col, rep_rho, rep_p in [("path_tortuosity",     -0.71, 0.001),
                                ("n_exit_v3",           +0.56, 0.007),
                                ("mean_handling_time_s",+0.59, 0.003)]:
        sig = "p<0.001" if rep_p < 0.001 else f"p = {rep_p:.3f}"
        print(f"    {col:25s}  rho = {rep_rho:+.2f}  ({sig})")
else:
    for col, rep_rho, rep_p in [("path_tortuosity",     -0.71, 0.001),
                                ("n_exit_v3",           +0.56, 0.007),
                                ("mean_handling_time_s",+0.59, 0.003)]:
        rho, p, n = spearman(sub_on["mean_dbm"].values, sub_on[col].values)
        s_rho = flag(rep_rho, rho, 0.05)
        s_p = "p<0.001" if rep_p < 0.001 else f"{rep_p:.3f}"
        print(f"  {col:25s}  report rho = {rep_rho:+.2f} (p {s_p})  "
              f"computed rho = {rho:+.3f}, p = {p:.3f}, n = {n}   {s_rho}")


§4.10 — Spearman rho (received power vs indicators, sys 900 ON days):

  !! WARNING: mean_dbm column is entirely NaN in indicators_daily.
     The dBm loader cells in multi_day_pipeline_v3.ipynb need to be
     re-run (they probably failed or were skipped when the v3 pipeline
     was last re-run). Until that happens the §4.10 numbers cannot be
     verified from this notebook.
  report values (cannot be checked yet):
    path_tortuosity            rho = -0.71  (p = 0.001)
    n_exit_v3                  rho = +0.56  (p = 0.007)
    mean_handling_time_s       rho = +0.59  (p = 0.003)


## §4.11 — Foraging Impairment Index (FII)

Report (around line ≈1100):
- Pooled FII: rank-biserial $r = +0.08$
- 95 % bootstrap CI: $[+0.01,\, +0.36]$
- $n_{\mathrm{ON}} = 36$, $n_{\mathrm{OFF}} = 34$ day-system rows
- Verdict: **not detected** (no individual indicator survives BH)

z-score reference = BASELINE + OFF distribution per system.
Sign-aligned mean of the six standardised indicators.


In [11]:
# ─── §4.11 verification: pooled FII + bootstrap CI ───
INDICATORS = ["neg_exit_count","neg_re_ratio","path_tortuosity","ifi_cv",
              "mean_handling_time_s","n_distinct_flowers"]

# Build z-scored indicators per system against (BASELINE + OFF) reference
ind_z = ind.copy()
for sid in [900, 939]:
    ref = ind_z[(ind_z.system_id==sid) & (ind_z.condition.isin(["BASELINE","OFF"]))]
    for col in INDICATORS:
        mu, sd = ref[col].mean(), ref[col].std(ddof=1)
        ind_z.loc[ind_z.system_id==sid, col + "_z"] = (
            (ind_z.loc[ind_z.system_id==sid, col] - mu) / sd)
ind_z["fii"] = ind_z[[c + "_z" for c in INDICATORS]].mean(axis=1)

pool = ind_z[ind_z.condition.isin(["ON","OFF"])]
on  = pool[pool.condition=="ON" ]["fii"].dropna().values
off = pool[pool.condition=="OFF"]["fii"].dropna().values

U, p, r = mwu(on, off)
ci_lo, ci_hi = bootstrap_r(on, off, n_boot=2000, seed=42)

print("§4.11 — Pooled FII verification:\n")
print(f"  n_ON  = {len(on)}   (report says 36)   {flag(36, len(on), 0)}")
print(f"  n_OFF = {len(off)}   (report says 34)   {flag(34, len(off), 0)}")
print(f"  pooled r       = {r:+.3f}  (report says +0.08)   {flag(+0.08, r, 0.03)}")
print(f"  raw p          = {p:.4f}")
print(f"  95% boot CI    = [{ci_lo:+.3f}, {ci_hi:+.3f}]  (report says [+0.01, +0.36])")
print(f"    CI low       : {flag(+0.01, ci_lo, 0.05)}")
print(f"    CI high      : {flag(+0.36, ci_hi, 0.05)}")
print()
print(f"  Verdict        : pooled r straddles or rests near 0 -> 'not detected'")


§4.11 — Pooled FII verification:

  n_ON  = 36   (report says 36)   MATCH
  n_OFF = 34   (report says 34)   MATCH
  pooled r       = +0.243  (report says +0.08)   MISMATCH
  raw p          = 0.0800
  95% boot CI    = [+0.026, +0.490]  (report says [+0.01, +0.36])
    CI low       : MATCH
    CI high      : MISMATCH

  Verdict        : pooled r straddles or rests near 0 -> 'not detected'


## Summary

After running all cells above, scan the output for `MISMATCH`. Every
hit corresponds to a number in §4 of `main4.tex` that no longer
matches the data. Update the report text and re-run this notebook to
verify the fix.

Note that some MISMATCHes are expected after a methodological change
(e.g. switching the Check 4 cone from per-bee to fixed +z will
change every indicator that depends on `hive_exit_v3`). In that
case, the report needs updating, not the code.